# Stage 2b — Legal Evaluation Set Expansion
## توسيع مجموعة التقييم القانونية (توليد آلي + تحقّق استرجاع)

يولّد هذا الدفتر أسئلة تقييم قانونية إضافية من مواد نظام العمل المفهرسة، لرفع حجم التقييم من 24 سؤالاً.

**المنهجية:** لكل مادة ← يولّد ALLaM سؤالاً طبيعياً (المادة هي الذهب) ← يُنظَّف من ذكر رقم المادة (منع تسريب) ← يتحقّق أن المادة الذهبية قابلة للاسترجاع ضمن أعلى 5.

> كل الأسئلة المُولّدة تُعلَّم pending_expert_review — يجب مراجعتها من خبير قانوني قبل الاعتماد النهائي.

In [1]:
# 2b.1 - Setup and load indexed legal articles
from pathlib import Path
import pandas as pd, numpy as np, re, torch
PROJECT_DIR = Path("saudi_labor_law_voice_agent_project")
CHUNKS_DIR = PROJECT_DIR / "04_chunks"
OUT_DIR = PROJECT_DIR / "03_final"
RNG = np.random.default_rng(7)

struct = pd.read_csv(CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv", encoding="utf-8-sig")
legal = struct[struct["source_type"] == "legal_article"].copy()
legal["_anum"] = pd.to_numeric(legal.get("article_number_int", legal["article_number"]), errors="coerce")
legal = legal.dropna(subset=["_anum"]).drop_duplicates(subset=["_anum"])
legal = legal[legal["chunk_text"].astype(str).str.len() > 250].reset_index(drop=True)
print(f"Substantive unique legal articles available for generation: {len(legal)}")

Substantive unique legal articles available for generation: 202


In [2]:
# 2b.2 - Load ALLaM generator and define question generation (+ leakage cleanup)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
GEN_PATH = r"C:/models/ALLaM-7B-Instruct-preview"
_qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
_gt = AutoTokenizer.from_pretrained(GEN_PATH)
_gm = AutoModelForCausalLM.from_pretrained(GEN_PATH, device_map="auto", quantization_config=_qc); _gm.eval()

def _strip_article_mentions(q):
    q = re.sub(r"(وفق\w*|حسب|بموجب|طبق\w*)?\s*للمادة\s*(رقم\s*)?\d+", "", q)
    q = re.sub(r"المادة\s*(رقم\s*)?\d+", "", q)
    q = re.sub(r"\s+", " ", q).strip(" ،,.-").strip()
    return q

def generate_question(article_text):
    sys = "أنت مساعد يصوغ أسئلة تقييم لنظام العمل السعودي."
    user = ("اقرأ المادة التالية، ثم صُغ سؤالاً واحداً طبيعياً قد يطرحه موظف أو صاحب عمل، "
            "بحيث تكون هذه المادة هي الإجابة عليه. لا تذكر رقم المادة في السؤال، "
            "ولا تنسخ نص المادة حرفياً، واجعله سؤالاً واقعياً قصيراً.\n\n"
            f"المادة:\n{str(article_text)[:900]}\n\nالسؤال فقط:")
    msgs = [{"role": "system", "content": sys}, {"role": "user", "content": user}]
    txt = _gt.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = _gt(txt, return_tensors="pt", truncation=True, max_length=1400).to(_gm.device)
    with torch.no_grad():
        out = _gm.generate(**inp, max_new_tokens=60, do_sample=False, pad_token_id=_gt.eos_token_id)
    q = _gt.decode(out[0][inp["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    q = q.split("\n")[0].strip().strip(chr(34)).strip()
    return _strip_article_mentions(q)

print("ALLaM question generator ready.")

C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W0621 11:04:30.777000 62292 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/291 [00:00<00:45,  6.31it/s]

Loading weights:   1%|          | 2/291 [00:00<00:44,  6.57it/s]

...

Loading weights:  91%|█████████▏| 266/291 [00:03<00:00, 74.11it/s]

Loading weights:  95%|█████████▍| 275/291 [00:04<00:00, 74.43it/s]

Loading weights:  98%|█████████▊| 284/291 [00:04<00:00, 74.52it/s]

Loading weights: 100%|██████████| 291/291 [00:04<00:00, 67.78it/s]

ALLaM question generator ready.


In [3]:
# 2b.3 - Load retrieval (best config) and define gold retrievability check
from sentence_transformers import SentenceTransformer, CrossEncoder
import chromadb
_dev = "cuda" if torch.cuda.is_available() else "cpu"
_emb = SentenceTransformer(r"C:/models/multilingual-e5-large", device=_dev)
_rer = CrossEncoder(r"C:/models/bge-reranker-v2-m3", device=_dev)
_col = chromadb.PersistentClient(path=str(PROJECT_DIR / "06_chroma_db")).get_collection("rag_structural_e5_large")

def gold_retrievable(question, gold_anum, k=5):
    qv = _emb.encode("query: " + str(question), normalize_embeddings=True).tolist()
    res = _col.query(query_embeddings=[qv], n_results=30)
    docs = res["documents"][0]
    if not docs:
        return False, None
    sc = _rer.predict([[str(question), d] for d in docs]); order = np.argsort(sc)[::-1][:k]
    for rank, idx in enumerate(order, 1):
        m = re.search(r"رقم المادة:\s*(\d+)", docs[idx])
        if m and int(m.group(1)) == int(gold_anum):
            return True, rank
    return False, None

print("Retrieval validator ready.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 10136.74it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 10977.51it/s]

Retrieval validator ready.


In [4]:
# 2b.4 - Generate + validate questions for all substantive articles
records = []
for i, r in legal.iterrows():
    anum = int(r["_anum"])
    q = generate_question(r["chunk_text"])
    if len(q) < 12:
        continue
    ok, rank = gold_retrievable(q, anum, k=5)
    records.append({
        "eval_id": f"legal_autogen_{anum}", "question": q,
        "eval_type": "legal_article_retrieval", "gold_article_number": anum,
        "gold_article_numbers": str(anum), "is_out_of_scope": False,
        "gold_retrievable_top5": ok, "gold_rank": rank,
        "source": "auto_generated_allam", "review_status": "pending_expert_review",
    })
    if (i + 1) % 25 == 0:
        print(f"  processed {i+1}/{len(legal)}")
autogen = pd.DataFrame(records)
print(f"\nGenerated {len(autogen)} questions | gold retrievable in top-5: "
      f"{int(autogen['gold_retrievable_top5'].sum())} ({autogen['gold_retrievable_top5'].mean()*100:.0f}%)")

C:\Users\PC\anaconda3\envs\datacolection\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  processed 25/202


  processed 50/202


  processed 75/202


  processed 100/202


  processed 125/202


  processed 150/202


  processed 175/202


  processed 200/202



Generated 202 questions | gold retrievable in top-5: 195 (97%)


In [5]:
# 2b.5 - Save expanded evaluation sets
autogen = autogen.drop_duplicates(subset=["question"]).reset_index(drop=True)
autogen.to_csv(OUT_DIR / "legal_eval_autogen_all.csv", index=False, encoding="utf-8-sig")

validated = autogen[autogen["gold_retrievable_top5"]].copy()
validated.to_csv(OUT_DIR / "legal_eval_autogen_validated.csv", index=False, encoding="utf-8-sig")

existing = pd.read_csv(OUT_DIR / "rag_evaluation_dataset.csv", encoding="utf-8-sig")
existing_legal = existing[existing["eval_type"] == "legal_article_retrieval"].copy()
existing_legal["source"] = "human_curated"; existing_legal["review_status"] = "approved"
common = ["question", "eval_type", "gold_article_number", "gold_article_numbers", "source", "review_status"]
merged = pd.concat(
    [existing_legal[[c for c in common if c in existing_legal.columns]], validated[common]],
    ignore_index=True, sort=False).drop_duplicates(subset=["question"])
merged.to_csv(OUT_DIR / "legal_eval_expanded_merged.csv", index=False, encoding="utf-8-sig")

print("=== Expansion summary ===")
print(f"  existing human-curated legal questions : {len(existing_legal)}")
print(f"  auto-generated (all)                   : {len(autogen)}")
print(f"  auto-generated validated (gold in top5): {len(validated)}")
print(f"  EXPANDED merged legal eval set         : {len(merged)}  (was {len(existing_legal)})")
print("\nFiles saved to", OUT_DIR)
merged.head(8)

=== Expansion summary ===
  existing human-curated legal questions : 24
  auto-generated (all)                   : 201
  auto-generated validated (gold in top5): 194
  EXPANDED merged legal eval set         : 218  (was 24)

Files saved to saudi_labor_law_voice_agent_project\03_final


,question,eval_type,gold_article_number,gold_article_numbers,source,review_status
0,كم مدة فترة التجربة في نظام العمل السعودي؟,legal_article_retrieval,53.0,53,human_curated,approved
1,هل يجوز وضع العامل تحت التجربة أكثر من مرة لدى...,legal_article_retrieval,54.0,54,human_curated,approved
2,ما الحد الأقصى لساعات العمل اليومية في نظام ال...,legal_article_retrieval,98.0,98,human_curated,approved
3,ما الحد الأقصى لساعات العمل الأسبوعية؟,legal_article_retrieval,98.0,98,human_curated,approved
4,هل تختلف ساعات العمل في شهر رمضان؟,legal_article_retrieval,98.0,98,human_curated,approved
5,ما مدة الإجازة السنوية المستحقة للعامل؟,legal_article_retrieval,109.0,109,human_curated,approved
6,متى تزيد الإجازة السنوية إلى ثلاثين يومًا؟,legal_article_retrieval,109.0,109,human_curated,approved
7,هل يستحق العامل أجرًا عن أيام الإجازة غير المس...,legal_article_retrieval,111.0,111,human_curated,approved


## 2b.6 — Retrieval performance on the expanded set (credible, large sample)
تقييم استرجاع موثوق على الـ201 سؤالاً المُولّدة مع فترات ثقة Bootstrap — يرفع المصداقية مقابل الـ24 الأصلية.

In [ ]:
# 2b.6 - Retrieval metrics on the expanded auto-generated legal set (from saved gold ranks)
ag = pd.read_csv(OUT_DIR / "legal_eval_autogen_all.csv", encoding="utf-8-sig")
rank = ag["gold_rank"]
metrics = {
    "Recall@1": (rank == 1).astype(float),
    "Recall@3": (rank <= 3).astype(float),
    "Recall@5": (rank <= 5).astype(float),
    "MRR":      rank.apply(lambda x: 1.0/x if pd.notna(x) else 0.0),
    "nDCG@5":   rank.apply(lambda x: 1.0/np.log2(x+1) if pd.notna(x) else 0.0),
}
def _ci(x, n=10000):
    x = np.asarray(x, float)
    boot = [RNG.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(boot, [2.5, 97.5])
rows = []
for k, v in metrics.items():
    lo, hi = _ci(v.values)
    rows.append({"metric": k, "n": len(ag), "value": round(float(v.mean()),4),
                 "ci95_low": round(lo,4), "ci95_high": round(hi,4)})
retrieval_metrics_expanded = pd.DataFrame(rows)
print("Retrieval on expanded auto-generated legal set (n=%d) | best config: structural+e5_large+hybrid+reranker" % len(ag))
print(retrieval_metrics_expanded.to_string(index=False))
print("\nNOTE: computed on ALL generated questions (incl. the 7 that missed top-5), so Recall@5 is honest, not selection-biased.")
retrieval_metrics_expanded